# SPRINT 2

- This notebook uses the cleaned output from Sprint 1 to create additional predictors based on the project hypotheses.
- This notebook only creates and verifies features. Imputation, encoding, scaling, train-validation splitting and model training will be done in another notebook

## Import packages
- Only pandas, NumPy and `Path` are required here. The scikit-learn preprocessing tools are intentionally kept in the modelling notebook.

In [44]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

## Load the cleaned output from Sprint 1

In [59]:
STORAGE_DIR = Path("../storage/cleaned")

cleaned_data_path = STORAGE_DIR / "cleaned.csv"

model_data_df = pd.read_csv(
    cleaned_data_path
)

print("Cleaned data shape:", model_data_df.shape)
model_data_df

Cleaned data shape: (26707, 38)


,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,doctor_recc_h1n1,doctor_recc_seasonal,chronic_med_condition,child_under_6_months,health_worker,health_insurance,opinion_h1n1_vacc_effective,opinion_h1n1_risk,opinion_h1n1_sick_from_vacc,opinion_seas_vacc_effective,opinion_seas_risk,opinion_seas_sick_from_vacc,age_group,education,race,sex,income_poverty,marital_status,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation,h1n1_vaccine,seasonal_vaccine
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,2.0,2.0,1.0,2.0,55 - 64 Years,< 12 Years,White,Female,Below Poverty,Not Married,Own,Not in Labor Force,oxchjgsf,Non-MSA,0.0,0.0,Unknown,Unknown,0,0
1,1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,4.0,4.0,4.0,2.0,4.0,35 - 44 Years,12 Years,White,Male,Below Poverty,Not Married,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0.0,0.0,pxcmvdjn,xgwztkwe,0,1
2,2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1.0,0.0,0.0,NaN,3.0,1.0,1.0,4.0,1.0,2.0,18 - 34 Years,College Graduate,White,Male,"<= $75,000, Above Poverty",Not Married,Own,Employed,qufhixun,"MSA, Not Principle City",2.0,0.0,rucpziij,xtkaffoo,0,0
3,3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,NaN,3.0,3.0,5.0,5.0,4.0,1.0,65+ Years,12 Years,White,Female,Below Poverty,Not Married,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0.0,0.0,Unknown,Unknown,0,1
4,4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,3.0,2.0,3.0,1.0,4.0,45 - 54 Years,Some College,White,Female,"<= $75,000, Above Poverty",Married,Own,Employed,qufhixun,"MSA, Not Principle City",1.0,0.0,wxleyezf,emcorrxb,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26702,26702,2.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,1.0,1.0,5.0,2.0,2.0,65+ Years,Some College,White,Female,"<= $75,000, Above Poverty",Not Married,Own,Not in Labor Force,qufhixun,Non-MSA,0.0,0.0,Unknown,Unknown,0,0
26703,26703,1.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,4.0,2.0,2.0,5.0,1.0,1.0,18 - 34 Years,College Graduate,White,Male,"<= $75,000, Above Poverty",Not Married,Rent,Employed,lzgpxyit,"MSA, Principle City",1.0,0.0,fcxhlnwr,cmhcxjea,0,0
26704,26704,2.0,2.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,4.0,4.0,2.0,5.0,4.0,2.0,55 - 64 Years,Some College,White,Female,Unknown,Not Married,Own,Unknown,lzgpxyit,"MSA, Not Principle City",0.0,0.0,Unknown,Unknown,0,1
26705,26705,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,2.0,2.0,1.0,2.0,18 - 34 Years,Some College,Hispanic,Female,"<= $75,000, Above Poverty",Married,Rent,Employed,lrircsnp,Non-MSA,1.0,0.0,fcxhlnwr,haliazsg,0,0


## Confirm that the columns required for feature engineering are present.

In [46]:
required_columns = [
    "respondent_id",
    "h1n1_vaccine",
    "seasonal_vaccine",
    "health_insurance",
    "doctor_recc_h1n1",
    "doctor_recc_seasonal",
    "income_poverty",
    "h1n1_concern",
    "opinion_h1n1_risk",
    "household_adults",
    "household_children"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in model_data_df.columns
]

print("Missing required columns:", missing_required_columns)

if missing_required_columns:
    raise ValueError(
        f"Required columns are missing: {missing_required_columns}"
    )

Missing required columns: []


## Prepare numeric columns

- Converts all binary, scale, and count columns into numeric values, changing non-numeric entries to NaN.

In [47]:
binary_columns = [
    "behavioral_antiviral_meds",
    "behavioral_avoidance",
    "behavioral_face_mask",
    "behavioral_wash_hands",
    "behavioral_large_gatherings",
    "behavioral_outside_home",
    "behavioral_touch_face",
    "doctor_recc_h1n1",
    "doctor_recc_seasonal",
    "chronic_med_condition",
    "child_under_6_months",
    "health_worker",
    "health_insurance"
]

scale_and_count_columns = [
    "h1n1_concern",
    "h1n1_knowledge",
    "opinion_h1n1_vacc_effective",
    "opinion_h1n1_risk",
    "opinion_h1n1_sick_from_vacc",
    "opinion_seas_vacc_effective",
    "opinion_seas_risk",
    "opinion_seas_sick_from_vacc",
    "household_adults",
    "household_children"
]

for column in binary_columns + scale_and_count_columns:
    model_data_df[column] = pd.to_numeric(
        model_data_df[column],
        errors="coerce"
    )

In [48]:
model_data_df.dtypes

respondent_id                    int64
h1n1_concern                   float64
h1n1_knowledge                 float64
behavioral_antiviral_meds      float64
behavioral_avoidance           float64
behavioral_face_mask           float64
behavioral_wash_hands          float64
behavioral_large_gatherings    float64
behavioral_outside_home        float64
behavioral_touch_face          float64
doctor_recc_h1n1               float64
doctor_recc_seasonal           float64
chronic_med_condition          float64
child_under_6_months           float64
health_worker                  float64
health_insurance               float64
opinion_h1n1_vacc_effective    float64
opinion_h1n1_risk              float64
opinion_h1n1_sick_from_vacc    float64
opinion_seas_vacc_effective    float64
opinion_seas_risk              float64
opinion_seas_sick_from_vacc    float64
age_group                          str
education                          str
race                               str
sex                      

# FEATURE ENGINEERING

| Engineering method | New features created | Count |
|---|---|---:|
| Missingness flags | `health_insurance_missing`, `doctor_recc_h1n1_missing`, `doctor_recc_seasonal_missing`, `income_poverty_missing` | 4 |
| Uncertainty flags | Six `_dont_know` columns | 6 |
| Aggregation | `preventive_behavior_score` | 1 |
| Normalization and averaging | `h1n1_risk_perception_score` | 1 |
| Direction adjustment and averaging | Two vaccine-confidence scores | 2 |
| Arithmetic combination | `household_size` | 1 |
| Binary/logical combination | `any_doctor_recommendation` | 1 |
| Missingness count | `n_missing_answers` | 1 |
| **Total** | **All engineered features** | **17** |

## Identify “Don't know” using the indicators


In [49]:
opinion_columns = [
    "opinion_h1n1_vacc_effective",
    "opinion_h1n1_risk",
    "opinion_h1n1_sick_from_vacc",
    "opinion_seas_vacc_effective",
    "opinion_seas_risk",
    "opinion_seas_sick_from_vacc"
]

for column in opinion_columns:
    model_data_df[f"{column}_dont_know"] = (
        model_data_df[column]
        .eq(3)
        .astype(int)
    )

model_data_df

,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,doctor_recc_h1n1,doctor_recc_seasonal,chronic_med_condition,child_under_6_months,health_worker,health_insurance,opinion_h1n1_vacc_effective,opinion_h1n1_risk,opinion_h1n1_sick_from_vacc,opinion_seas_vacc_effective,opinion_seas_risk,opinion_seas_sick_from_vacc,age_group,education,race,sex,income_poverty,marital_status,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation,h1n1_vaccine,seasonal_vaccine,opinion_h1n1_vacc_effective_dont_know,opinion_h1n1_risk_dont_know,opinion_h1n1_sick_from_vacc_dont_know,opinion_seas_vacc_effective_dont_know,opinion_seas_risk_dont_know,opinion_seas_sick_from_vacc_dont_know
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,2.0,2.0,1.0,2.0,55 - 64 Years,< 12 Years,White,Female,Below Poverty,Not Married,Own,Not in Labor Force,oxchjgsf,Non-MSA,0.0,0.0,Unknown,Unknown,0,0,1,0,0,0,0,0
1,1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,4.0,4.0,4.0,2.0,4.0,35 - 44 Years,12 Years,White,Male,Below Poverty,Not Married,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0.0,0.0,pxcmvdjn,xgwztkwe,0,1,0,0,0,0,0,0
2,2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1.0,0.0,0.0,NaN,3.0,1.0,1.0,4.0,1.0,2.0,18 - 34 Years,College Graduate,White,Male,"<= $75,000, Above Poverty",Not Married,Own,Employed,qufhixun,"MSA, Not Principle City",2.0,0.0,rucpziij,xtkaffoo,0,0,1,0,0,0,0,0
3,3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,NaN,3.0,3.0,5.0,5.0,4.0,1.0,65+ Years,12 Years,White,Female,Below Poverty,Not Married,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0.0,0.0,Unknown,Unknown,0,1,1,1,0,0,0,0
4,4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,3.0,2.0,3.0,1.0,4.0,45 - 54 Years,Some College,White,Female,"<= $75,000, Above Poverty",Married,Own,Employed,qufhixun,"MSA, Not Principle City",1.0,0.0,wxleyezf,emcorrxb,0,0,1,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26702,26702,2.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,1.0,1.0,5.0,2.0,2.0,65+ Years,Some College,White,Female,"<= $75,000, Above Poverty",Not Married,Own,Not in Labor Force,qufhixun,Non-MSA,0.0,0.0,Unknown,Unknown,0,0,1,0,0,0,0,0
26703,26703,1.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,4.0,2.0,2.0,5.0,1.0,1.0,18 - 34 Years,College Graduate,White,Male,"<= $75,000, Above Poverty",Not Married,Rent,Employed,lzgpxyit,"MSA, Principle City",1.0,0.0,fcxhlnwr,cmhcxjea,0,0,0,0,0,0,0,0
26704,26704,2.0,2.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,4.0,4.0,2.0,5.0,4.0,2.0,55 - 64 Years,Some College,White,Female,Unknown,Not Married,Own,Unknown,lzgpxyit,"MSA, Not Principle City",0.0,0.0,Unknown,Unknown,0,1,0,0,0,0,0,0
26705,26705,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,2.0,2.0,1.0,2.0,18 - 34 Years,Some College,Hispanic,Female,"<= $75,000, Above Poverty",Married,Rent,Employed,lrircsnp,Non-MSA,1.0,0.0,fcxhlnwr,haliazsg,0,0,1,0,0,0,0,0


## Preventive-behaviour score

- Seven binary preventive behaviours are added with equal weight. The score ranges from 0 to 7.

| Person | Calculation | Score |
|---|---|---:|
| A | 0 + 1 + 0 + 1 + 1 + 0 + 1 | 4 |
| B | 0 + 0 + 0 + 1 + 0 + 0 + 0 | 1 |
| C | 1 + 1 + 1 + 1 + 1 + 1 + 1 | 7 |

In [50]:
preventive_behavior_columns = [
    "behavioral_antiviral_meds",
    "behavioral_avoidance",
    "behavioral_face_mask",
    "behavioral_wash_hands",
    "behavioral_large_gatherings",
    "behavioral_outside_home",
    "behavioral_touch_face"
]

model_data_df["preventive_behavior_score"] = (
    model_data_df[preventive_behavior_columns]
    .sum(axis=1, min_count=1)
)

## H1N1 risk-perception score

- `h1n1_concern` uses a 0–3 scale and `opinion_h1n1_risk` uses a 1–5 scale. Min-max normalization places both on a 0–1 scale before averaging.
- Because `3` means “Don't know” in the risk question, it is temporarily changed to `NaN` for this calculation. The separate `_dont_know` indicator preserves that information.

In [51]:
# Normalize concern from 0-3 to 0-1.
normalized_h1n1_concern = (
    model_data_df["h1n1_concern"] / 3
)

# Exclude "Don't know" from the numerical risk score.
h1n1_risk_for_score = (
    model_data_df["opinion_h1n1_risk"]
    .mask(
        model_data_df["opinion_h1n1_risk"].eq(3)
    )
)

# Normalize valid risk answers from 1-5 to 0-1.
normalized_h1n1_risk = (
    h1n1_risk_for_score - 1
) / 4

# Average the available components for each respondent.
model_data_df["h1n1_risk_perception_score"] = (
    pd.concat(
        [
            normalized_h1n1_concern,
            normalized_h1n1_risk
        ],
        axis=1
    )
    .mean(axis=1)
)

model_data_df

,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,doctor_recc_h1n1,doctor_recc_seasonal,chronic_med_condition,child_under_6_months,health_worker,health_insurance,opinion_h1n1_vacc_effective,opinion_h1n1_risk,opinion_h1n1_sick_from_vacc,opinion_seas_vacc_effective,opinion_seas_risk,opinion_seas_sick_from_vacc,age_group,education,race,sex,income_poverty,marital_status,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation,h1n1_vaccine,seasonal_vaccine,opinion_h1n1_vacc_effective_dont_know,opinion_h1n1_risk_dont_know,opinion_h1n1_sick_from_vacc_dont_know,opinion_seas_vacc_effective_dont_know,opinion_seas_risk_dont_know,opinion_seas_sick_from_vacc_dont_know,preventive_behavior_score,h1n1_risk_perception_score
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,2.0,2.0,1.0,2.0,55 - 64 Years,< 12 Years,White,Female,Below Poverty,Not Married,Own,Not in Labor Force,oxchjgsf,Non-MSA,0.0,0.0,Unknown,Unknown,0,0,1,0,0,0,0,0,2.0,0.166667
1,1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,4.0,4.0,4.0,2.0,4.0,35 - 44 Years,12 Years,White,Male,Below Poverty,Not Married,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0.0,0.0,pxcmvdjn,xgwztkwe,0,1,0,0,0,0,0,0,4.0,0.875000
2,2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1.0,0.0,0.0,NaN,3.0,1.0,1.0,4.0,1.0,2.0,18 - 34 Years,College Graduate,White,Male,"<= $75,000, Above Poverty",Not Married,Own,Employed,qufhixun,"MSA, Not Principle City",2.0,0.0,rucpziij,xtkaffoo,0,0,1,0,0,0,0,0,1.0,0.166667
3,3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,NaN,3.0,3.0,5.0,5.0,4.0,1.0,65+ Years,12 Years,White,Female,Below Poverty,Not Married,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0.0,0.0,Unknown,Unknown,0,1,1,1,0,0,0,0,3.0,0.333333
4,4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,3.0,2.0,3.0,1.0,4.0,45 - 54 Years,Some College,White,Female,"<= $75,000, Above Poverty",Married,Own,Employed,qufhixun,"MSA, Not Principle City",1.0,0.0,wxleyezf,emcorrxb,0,0,1,1,0,1,0,0,4.0,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26702,26702,2.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,1.0,1.0,5.0,2.0,2.0,65+ Years,Some College,White,Female,"<= $75,000, Above Poverty",Not Married,Own,Not in Labor Force,qufhixun,Non-MSA,0.0,0.0,Unknown,Unknown,0,0,1,0,0,0,0,0,2.0,0.333333
26703,26703,1.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,4.0,2.0,2.0,5.0,1.0,1.0,18 - 34 Years,College Graduate,White,Male,"<= $75,000, Above Poverty",Not Married,Rent,Employed,lzgpxyit,"MSA, Principle City",1.0,0.0,fcxhlnwr,cmhcxjea,0,0,0,0,0,0,0,0,2.0,0.291667
26704,26704,2.0,2.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,4.0,4.0,2.0,5.0,4.0,2.0,55 - 64 Years,Some College,White,Female,Unknown,Not Married,Own,Unknown,lzgpxyit,"MSA, Not Principle City",0.0,0.0,Unknown,Unknown,0,1,0,0,0,0,0,0,5.0,0.708333
26705,26705,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,2.0,2.0,1.0,2.0,18 - 34 Years,Some College,Hispanic,Female,"<= $75,000, Above Poverty",Married,Rent,Employed,lrircsnp,Non-MSA,1.0,0.0,fcxhlnwr,haliazsg,0,0,1,0,0,0,0,0,1.0,0.166667


## Vaccine-confidence scores

- This creates H1N1 and seasonal vaccine-confidence scores by combining perceived effectiveness with reversed sickness worry, while excluding “Don’t know” responses.
- It uses min-max normalization (scaling) to convert the opinion values to 0–1, then reverses the worry scale and averages the two components into a confidence score.

In [52]:
def opinion_for_score(column):
    # Return the opinion values with 3 ("Don't know") treated as missing.
    return model_data_df[column].mask(
        model_data_df[column].eq(3)
    )


# H1N1 vaccine confidence.
normalized_h1n1_effectiveness = (
    opinion_for_score("opinion_h1n1_vacc_effective") - 1
) / 4

reversed_h1n1_sickness_fear = (
    5 - opinion_for_score("opinion_h1n1_sick_from_vacc")
) / 4

model_data_df["h1n1_vaccine_confidence_score"] = (
    pd.concat(
        [
            normalized_h1n1_effectiveness,
            reversed_h1n1_sickness_fear
        ],
        axis=1
    )
    .mean(axis=1)
)

# Seasonal vaccine confidence.
normalized_seasonal_effectiveness = (
    opinion_for_score("opinion_seas_vacc_effective") - 1
) / 4

reversed_seasonal_sickness_fear = (
    5 - opinion_for_score("opinion_seas_sick_from_vacc")
) / 4

model_data_df["seasonal_vaccine_confidence_score"] = (
    pd.concat(
        [
            normalized_seasonal_effectiveness,
            reversed_seasonal_sickness_fear
        ],
        axis=1
    )
    .mean(axis=1)
)

## Household-size and doctor-recommendation features

- Household size combines other adults, children and the respondent.
- `any_doctor_recommendation` equals `1` when at least one vaccine was recommended.
- If both doctor-recommendation answers are missing, the combined feature remains missing.

In [53]:
# The source columns count other household members, so add 1
# to include the respondent.
model_data_df["household_size"] = (
    model_data_df["household_adults"]
    + model_data_df["household_children"]
    + 1
)

# Record whether a doctor recommended at least one vaccine.
model_data_df["any_doctor_recommendation"] = (
    model_data_df[
        [
            "doctor_recc_h1n1",
            "doctor_recc_seasonal"
        ]
    ]
    .max(axis=1, skipna=True)
)

# Preserve missingness when neither source answer is known.
both_doctor_answers_missing = model_data_df[
    ["doctor_recc_h1n1", "doctor_recc_seasonal"]
].isna().all(axis=1)

model_data_df.loc[
    both_doctor_answers_missing,
    "any_doctor_recommendation"
] = np.nan

# Verify the engineered features

These checks confirm that all 17 engineered columns exist, display their summary statistics and missing values, and verify the expected ranges of the main scores.

In [54]:
# The 12 engineered features that we decided to retain.
engineered_features = [
    *[f"{column}_dont_know" for column in opinion_columns],
    "preventive_behavior_score",
    "h1n1_risk_perception_score",
    "h1n1_vaccine_confidence_score",
    "seasonal_vaccine_confidence_score",
    "household_size",
    "any_doctor_recommendation"
]

# Check whether any expected engineered feature was not created.
missing_engineered_features = [
    feature
    for feature in engineered_features
    if feature not in model_data_df.columns
]

print("Expected engineered features:", 12)
print("Created engineered features:", len(engineered_features))
print("Missing engineered features:", missing_engineered_features)

# Stop only if an expected feature is missing.
if missing_engineered_features:
    raise ValueError(
        f"Engineered features are missing: {missing_engineered_features}"
    )

# Summarise the engineered features.
feature_summary = (
    model_data_df[engineered_features]
    .describe()
    .T
)

# Add the number of missing values in each engineered feature.
feature_summary["missing_count"] = (
    model_data_df[engineered_features]
    .isna()
    .sum()
)

display(feature_summary)

Expected engineered features: 12
Created engineered features: 12
Missing engineered features: []


,count,mean,std,min,25%,50%,75%,max,missing_count
opinion_h1n1_vacc_effective_dont_know,26707.0,0.176845,0.381545,0.0,0.000000,0.000000,0.000,1.0,0
opinion_h1n1_risk_dont_know,26707.0,0.041824,0.200191,0.0,0.000000,0.000000,0.000,1.0,0
opinion_h1n1_sick_from_vacc_dont_know,26707.0,0.005542,0.074237,0.0,0.000000,0.000000,0.000,1.0,0
opinion_seas_vacc_effective_dont_know,26707.0,0.045531,0.208470,0.0,0.000000,0.000000,0.000,1.0,0
opinion_seas_risk_dont_know,26707.0,0.025349,0.157186,0.0,0.000000,0.000000,0.000,1.0,0
opinion_seas_sick_from_vacc_dont_know,26707.0,0.003520,0.059223,0.0,0.000000,0.000000,0.000,1.0,0
preventive_behavior_score,26707.0,3.043846,1.599161,0.0,2.000000,3.000000,4.000,7.0,0
h1n1_risk_perception_score,26707.0,0.438822,0.264432,0.0,0.291667,0.458333,0.625,1.0,0
h1n1_vaccine_confidence_score,26601.0,0.704579,0.234169,0.0,0.500000,0.750000,0.875,1.0,106
seasonal_vaccine_confidence_score,26652.0,0.744953,0.221353,0.0,0.625000,0.750000,0.875,1.0,55


In [55]:
# Expected ranges based on how the features were constructed.
range_checks = {
    "preventive_behavior_score": (0, 7),
    "h1n1_risk_perception_score": (0, 1),
    "h1n1_vaccine_confidence_score": (0, 1),
    "seasonal_vaccine_confidence_score": (0, 1),
    "household_size": (1, 7),
    "any_doctor_recommendation": (0, 1)
}

range_results = {}

for feature, (minimum, maximum) in range_checks.items():
    is_valid = (
        model_data_df[feature]
        .dropna()
        .between(minimum, maximum)
        .all()
    )

    range_results[feature] = is_valid
    print(f"{feature}: {is_valid}")

if not all(range_results.values()):
    raise ValueError(
        "At least one engineered feature contains values outside its expected range."
    )

preventive_behavior_score: True
h1n1_risk_perception_score: True
h1n1_vaccine_confidence_score: True
seasonal_vaccine_confidence_score: True
household_size: True
any_doctor_recommendation: True


## Save the feature-engineered dataset

The saved file contains:
- respondent ID;
- original predictors;
- 17 engineered predictors;
- both target variables.

In [61]:
STORAGE_DIR2 = Path("../storage/feature_engineered")
engineered_data_path = (
    STORAGE_DIR2 / "cleaned_and_engineered.csv"
)

 
model_data_df.to_csv(
    engineered_data_path,
    index=False
)

print("Feature-engineered data saved successfully.")
print("Saved to:", engineered_data_path)
print("Saved shape:", model_data_df.shape)

Feature-engineered data saved successfully.
Saved to: ../storage/feature_engineered/cleaned_and_engineered.csv
Saved shape: (26707, 38)


## Next step

Use `storage/train_engineered.csv` as the input to `03_modeling.ipynb`.

The modelling notebook will:

1. separate predictors and targets;
2. create the train-validation split;
3. define numeric, ordinal and nominal preprocessing;
4. train baseline models;
5. apply cross-validation and hyperparameter tuning;
6. evaluate and interpret the final models.